In [22]:
print('Đang khai báo thư viện')

import joblib
import pandas as pd
import os
from sklearn.metrics import accuracy_score, f1_score, classification_report

print('Khai báo thư viện thành công')

Đang khai báo thư viện
Khai báo thư viện thành công


In [23]:
model_path = os.path.join("..", "Model", "random_forest_model.pkl")
feature_path = os.path.join("..", "Model", "feature_names.pkl")

rf = joblib.load(model_path)

feature_names = joblib.load(feature_path)

print("Đã load Random Forest model")
print("Đã load feature names")

Đã load Random Forest model
Đã load feature names


In [25]:
# DỰ ĐOÁN TRÊN TẬP TEST

# Load test data (giả sử bạn có file test_cleaned.csv)
test_path = os.path.join("..", "Data", "DataCleaned", "test_cleaned.csv")
df_test = pd.read_csv(test_path)

print(f"Test shape: {df_test.shape}")

# Xử lý giống như train/val
drop_cols = ['Id', 'Artist Name', 'Track Name']
existing_drop_test = [col for col in drop_cols if col in df_test.columns]

# Giữ lại Class nếu có (nếu test có nhãn để đánh giá)
if 'Class' in df_test.columns:
    X_test = df_test.drop(columns=['Class'] + existing_drop_test)
    y_test = df_test['Class']
    has_label = True
else:
    X_test = df_test.drop(columns=existing_drop_test)
    has_label = False

print(f"X_test shape: {X_test.shape}")

Test shape: (3600, 16)
X_test shape: (3600, 16)


In [27]:
# Dự đoán
y_pred_test = rf.predict(X_test)

# Xuất kết quả
print("\n=== KẾT QUẢ DỰ ĐOÁN TRÊN TEST ===")
print(f"Số lượng mẫu test: {len(y_pred_test)}")
print(f"Phân bố class dự đoán:\n{pd.Series(y_pred_test).value_counts().sort_index()}")

# Nếu có nhãn thật, đánh giá
if has_label:
    print(f"\nAccuracy Test  : {accuracy_score(y_test, y_pred_test):.4f}")
    print(f"Macro F1 Test  : {f1_score(y_test, y_pred_test, average='macro', zero_division=0):.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred_test, zero_division=0))

# Xuất file submission
submission = pd.DataFrame({
    'Id': df_test['Id'] if 'Id' in df_test.columns else range(len(y_pred_test)),
    'Class': y_pred_test
})
submission.to_csv('submission_rf.csv', index=False)
print("\nĐã lưu kết quả vào submission_rf.csv")


=== KẾT QUẢ DỰ ĐOÁN TRÊN TEST ===
Số lượng mẫu test: 3600
Phân bố class dự đoán:
0      128
1      136
2      291
3       79
4       54
5      286
6      427
7      143
8      351
9      614
10    1091
Name: count, dtype: int64

Đã lưu kết quả vào submission_rf.csv


In [34]:
# Gán ID cho mẫu test sau khi train để áp vào kết quả chạy trên Kaggle đánh giá điểm

sample_path = os.path.join("..", "Data", "sample_submission.csv")
sample = pd.read_csv(sample_path)

print(f"Sample shape: {sample.shape}")
print(f"Sample Id range: {sample['Id'].min()} - {sample['Id'].max()}")
print(f"First 10 Ids: {sample['Id'].head(10).tolist()}")

# Lấy đúng Id từ sample
correct_ids = sample['Id'].values

print(f"Number of predictions: {len(y_pred_test)}")

# Tạo submission với Id đúng
submission_fixed = pd.DataFrame({
    'Id': correct_ids,
    'Class': y_pred_test 
})

# Lưu file
output_path = os.path.join("submission_fixed.csv")
submission_fixed.to_csv(output_path, index=False)

print(f"\nSaved: {output_path}")
print(f"Submission shape: {submission_fixed.shape}")
print(f"Id range: {submission_fixed['Id'].min()} - {submission_fixed['Id'].max()}")

Sample shape: (3600, 2)
Sample Id range: 14397 - 17996
First 10 Ids: [14397, 14398, 14399, 14400, 14401, 14402, 14403, 14404, 14405, 14406]
Number of predictions: 3600

✅ Saved: submission_fixed.csv
Submission shape: (3600, 2)
Id range: 14397 - 17996
